# שבוע 6: ניתוח רכיבים עיקריים (PCA)

בשיעור זה נלמד:
- כיצד PCA מוצא את כיווני השונות העיקריים
- כיצד לצייר מרחב צורות
- כיצד לפרש את ציר PC1 ו-PC2
- חזרה על צורות בקצוות הצירים

> **הוראות**: הריצו כל תא בסדר מלמעלה למטה.

In [ ]:
!pip install morphops python-bidi -q
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
rtl = get_display
print('הכל מוכן!')

In [ ]:
import urllib.request
import morphops as mops

def parse_tps(text):
    specimens, ids = [], []
    lines = text.strip().split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('LM='):
            n_lm = int(line.split('=')[1])
            coords = []
            for j in range(n_lm):
                i += 1
                parts = lines[i].strip().replace(',', '.').split()
                coords.append([float(parts[0]), float(parts[1])])
            specimens.append(np.array(coords))
        elif line.startswith('ID='):
            ids.append(line.split('=')[1])
        i += 1
    return np.array(specimens), ids

def make_synthetic_coins(n, seed_offset=0):
    np.random.seed(42 + seed_offset)
    coins = []
    for i in range(n):
        cx, cy = np.random.uniform(100, 500), np.random.uniform(100, 500)
        scale = np.random.uniform(0.8, 1.2)
        ao = np.random.uniform(0, 0.2)
        lm = [[cx + (80*scale + np.random.randn()*3)*np.cos(2*np.pi*j/8 + ao),
               cy + (80*scale + np.random.randn()*3)*np.sin(2*np.pi*j/8 + ao)] for j in range(8)]
        coins.append(np.array(lm))
    return np.array(coins), [f'coin_{i+1:03d}' for i in range(n)]

base = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/coins/'
try:
    with urllib.request.urlopen(base + 'hadrian.tps') as r:
        lm_h, ids_h = parse_tps(r.read().decode('utf-8'))
    with urllib.request.urlopen(base + 'antoninus.tps') as r:
        lm_a, ids_a = parse_tps(r.read().decode('utf-8'))
except Exception as e:
    print(f'משתמשים בנתוני דוגמה: {e}')
    lm_h, ids_h = make_synthetic_coins(20, 0)
    lm_a, ids_a = make_synthetic_coins(15, 10)

all_lm = np.concatenate([lm_h, lm_a])
labels = np.array(['הדריאנוס'] * len(lm_h) + ['אנטונינוס פיוס'] * len(lm_a))
aligned, mean_shape, _ = mops.procrustes(all_lm)
print(f'GPA הושלם: {len(aligned)} מטבעות, {aligned.shape[1]} ציוני דרך')

## PCA על קואורדינטות פרוקרוסטס

אחרי GPA, כל מטבע מיוצג כנקודה במרחב בן 16 ממדים (8 ציוני דרך × 2 קואורדינטות).
PCA מוצא את הצירים שבהם השונות הגדולה ביותר.

In [ ]:
from sklearn.decomposition import PCA

# שטוח קואורדינטות: (n_specimens, n_landmarks * 2)
X = aligned.reshape(len(aligned), -1)

pca = PCA()
scores = pca.fit_transform(X)

# אחוז שונות לכל PC
var_explained = pca.explained_variance_ratio_ * 100
print('אחוז שונות מוסבר:')
for i, v in enumerate(var_explained[:6]):
    bar = '█' * int(v)
    print(f'  PC{i+1}: {v:5.1f}%  {bar}')

In [ ]:
# גרף מדרגות (Scree plot)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 8), var_explained[:7], color='steelblue', alpha=0.8)
ax.set_xlabel(rtl('רכיב עיקרי (PC)'))
ax.set_ylabel(rtl('שונות מוסברת (%)'))
ax.set_title(rtl('גרף מדרגות (Scree Plot) — מטבעות'))
ax.set_xticks(range(1, 8))
plt.tight_layout()
plt.show()

In [ ]:
# גרף פיזור PCA
colors = {'הדריאנוס': 'steelblue', 'אנטונינוס פיוס': 'coral'}
fig, ax = plt.subplots(figsize=(9, 7))

for group, color in colors.items():
    mask = labels == group
    ax.scatter(scores[mask, 0], scores[mask, 1], c=color, s=80, alpha=0.8,
               label=rtl(group), edgecolors='white', linewidth=0.5)

ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel(rtl(f'PC1 ({var_explained[0]:.1f}% שונות)'))
ax.set_ylabel(rtl(f'PC2 ({var_explained[1]:.1f}% שונות)'))
ax.set_title(rtl('מרחב צורות — מטבעות רומיים'), fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## חזרה על צורות בקצוות הצירים

כדי לפרש מה PC1 מייצג, נצייר את צורת הממוצע עם תזוזות חיוביות ושליליות לאורך PC1.

In [ ]:
def shape_at_pc(pca, mean_shape, pc_idx, scale):
    """צורה בנקודה מסוימת על ציר PC"""
    vec = pca.components_[pc_idx]
    displaced = mean_shape.flatten() + scale * vec
    return displaced.reshape(mean_shape.shape)

# כמה לנוע לאורך PC1 (בסטיות תקן)
sd_pc1 = np.std(scores[:, 0])
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = [rtl('PC1 שלילי (−2SD)'), rtl('צורת ממוצע'), rtl('PC1 חיובי (+2SD)')]
scales = [-2 * sd_pc1, 0, 2 * sd_pc1]

for ax, title, scale in zip(axes, titles, scales):
    shape = shape_at_pc(pca, mean_shape, 0, scale)
    ax.plot(shape[:, 0], shape[:, 1], 'o-', color='steelblue', markersize=8, linewidth=2)
    for j, (x, y) in enumerate(shape):
        ax.annotate(str(j+1), (x, y), textcoords='offset points', xytext=(4, 4), fontsize=9)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.45, 0.45)
    ax.set_ylim(-0.45, 0.45)

plt.suptitle(rtl('שינוי צורה לאורך PC1'), fontsize=14)
plt.tight_layout()
plt.show()

## תרגיל

1. כמה אחוז שונות מוסברת על ידי PC1 ו-PC2 יחד?
2. האם שתי הקבוצות (הדריאנוס / אנטונינוס פיוס) נפרדות בבירור במרחב PCA?
3. תנו תיאור מילולי: מה משתנה בצורת המטבע מקצה שמאלי לקצה ימני של PC1?
4. שנו `pc_idx=1` בפונקציה `shape_at_pc` כדי לראות מה PC2 מייצג.